### Задание 1
Счётчик

Определите класс Counter, реализующий положительный целочисленный счетчик, который может увеличивать или уменьшать свое значение в заданном диапазоне, включая границы диапазона. 

В классе должны быть предусмотрены следующие возможности:
- конструктор для инициализации счетчика значениями по умолчанию (стартовое значение и верхняя границы диапазона)
- метод для установки произвольного значения счётчика
- методы для увеличения и уменьшения текущего значения счетчика, по умолчанию на 1
- метод для получения текущего значения счётчика
- метод для сброса счётчика

Все методы класса должны проверять выход текущего значения счетчика за допустимый диапазон. 

Создайте экземпляр счетчика со значениями по умолчанию и выведите на экран его начальные параметры. Далее проверьте его работу циклом в пределах диапазона, увеличивая его текущее значение от минимально возможного до максимального. Протестируйте работу всех методов

Добавьте возможность складывать 2 экземпляра класса Counter, таким образом чтобы результатом сложения также был объект типа Counter

In [23]:
class Counter:
    def __init__(self, count: int = 0, end: int = 0):
        self._start = 1

        assert end > 0
        self._end = end

        assert count in range(self._start, self._end + 1), "Счетчик выходит за границы диапазона"
        self._count = count

    def set(self, count: int):
        self._count = count

    def incr(self, offset: int = 1):
        assert self._count + offset in range(self._start, self._end + 1), "Счетчик выходит за границы диапазона"
        self._count += offset
    
    def decr(self, offset: int = 1):
        assert self._count - offset in range(self._start, self._end + 1), "Счетчик выходит за границы диапазона"
        self._count -= offset

    def get(self):
        return self._count
    
    @property
    def stop(self):
        self._count = 1
    
    def __add__(self, other):
        return Counter(min([self._count, other._count]), max([self._end + other._end]))

counter = Counter(19, 30)
print(counter.__dict__)
counter.stop
print(counter.get(), end=" ")
for _ in range(1, 30):
    counter.incr(1)
    print(counter.get(), end=" ")

{'_start': 1, '_end': 30, '_count': 19}
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 

In [24]:
c1 = Counter(10, 20)
c2 = Counter(15, 30)
c1.incr()
c2.decr(5)
c3 = c1 + c2
c3.get(), c3.stop

(10, None)

In [25]:
while True:
    c3.incr(2)

AssertionError: Счетчик выходит за границы диапазона

In [26]:
c3.get()

49

### Задание 2

Аннотируйте задание с фигурами из предыдущего урока

In [27]:
import math

class Rectangle:
    def __init__(self, a: int | float, b: int | float):
        self.a = a
        self.b = b
    
    def get_perimeter(self):
        return (self.a + self.b) * 2
    
    def get_square(self):
        return self.a * self.b
    

class Circle:
    def __init__(self, r: int | float):
        self.r = r
    
    def get_perimeter(self):
        return self.r**2 * math.pi
    
    def get_square(self):
        return 2 * self.r * math.pi
    

class Rhombus:
    def __init__(self, p: int | float, q: int | float):
        self.p = p
        self.q = q
    
    def get_perimeter(self):
        return 2 * math.sqrt(self.p**2 + self.q**2)

    def get_square(self):
        return self.p * self.q / 2
    

def calculate_perimeter(figure):
    return figure.get_perimeter()

def calculate_square(figure):
    return figure.get_square()

rect = Rectangle(1, 2)
circ = Circle(2)
rhomb = Rhombus(1, 2)

print(*[calculate_perimeter(figure) for figure in [rect, circ, rhomb]])
print(*[calculate_square(figure) for figure in [rect, circ, rhomb]])

6 12.566370614359172 4.47213595499958
2 12.566370614359172 1.0


### Задание 3
Задание: Система бронирования коворкингов

Вы разрабатываете сервис бронирования рабочих мест в коворкингах. Фишка: пользователи могут обмениваться забронированными местами, если их планы поменялись.

Все классы должны быть датаклассами.

In [43]:
import datetime
from dataclasses import dataclass, field, replace
from enum import Enum
from typing import Optional

# 1. Композиция: местоположение
@dataclass
class Address:
    city: str
    street: str
    floor: int

# 2. Композиция: пользователь
@dataclass
class User:
    id: int
    name: str
    rating: float  # рейтинг от 1 до 5
    bonus_points: int  # бонусные баллы для компенсации

class WorkspaceType(Enum):
    DESK = "рабочее место"
    PRIVATE_OFFICE = "личный кабинет"
    MEETING_ROOM = "переговорная"

@dataclass
class Booking:
    """
    Бронирование рабочего места
    """
    # Композиция: бронирование имеет адрес и пользователя
    address: Address
    user: User
    workspace_type: WorkspaceType
    price_per_hour: int
    date: datetime.date
    hours: int  # количество часов

    booking_status: bool = field(init=False, default=True)
    create_at: datetime.datetime = field(init=False, default_factory=lambda: datetime.datetime.now())
    id: int = field(init=False, default_factory=lambda: abs(hash(datetime.datetime.now())))

    def total_price(self):
        return self.price_per_hour * self.hours
    
    def cancel(self):
        self.booking_status = False

    def change_user(self, new_user):
        self.user = new_user
    
    def __post_init__(self):
        assert self.hours > 0, "Указано неверное число часов"
        assert 1 <= self.user.rating <= 5, "У пользователя указан неверный рейтинг"

# Функция для обмена бронями между пользователями
def exchange_bookings(booking1: Booking, booking2: Booking) -> bool:
    """
    Обменивает брони между пользователями.
    Если цены разные, разница списывается/начисляется бонусными баллами.
    
    TODO: Реализовать логику:
    1. Проверить, что обе брони активны
    2. Проверить, что даты броней совпадают (для простоты)
    3. Обменять пользователей местами
    4. Компенсировать разницу в цене бонусными баллами
    5. Вернуть True, если обмен успешен
    """
    if not all([booking1.booking_status, booking2.booking_status]):
        return False
    
    if booking1.date != booking2.date:
        return False
    
    user1 = booking1.user
    user2 = booking2.user
    
    price1 = booking1.total_price()
    price2 = booking2.total_price()

    if not all([user1.bonus_points + price1 - price2 > 0, user2.bonus_points + price2 - price1 > 0]):
        return False

    user1.bonus_points += price1 - price2
    user2.bonus_points += price2 - price1
    
    booking1.user = user2
    booking2.user = user1


    return True


In [54]:
if __name__ == "__main__":
    # Создаем пользователей
    alice = User(id=1, name="Алиса", rating=4.8, bonus_points=500)
    bob = User(id=2, name="Боб", rating=4.2, bonus_points=200)
    
    # Создаем места
    desk = Address(city="Москва", street="Тверская, 15", floor=3)
    office = Address(city="Москва", street="Тверская, 15", floor=5)
    
    # Создаем брони
    booking1 = Booking(
        address=desk,
        user=alice,
        workspace_type=WorkspaceType.DESK,
        price_per_hour=300,
        date=datetime.date(2026, 3, 25),
        hours=4
    )
    
    booking2 = Booking(
        address=office,
        user=bob,
        workspace_type=WorkspaceType.PRIVATE_OFFICE,
        price_per_hour=1200,
        date=datetime.date(2026, 3, 25),
        hours=3
    )

    # TODO: Протестировать методы
    # - отмену брони
    # - смену пользователя
    # - обмен бронями с компенсацией

    print()
    print("Проверка отмены брони")
    booking1.cancel()
    print("booking1 статус:", booking1.booking_status)  # False

    print()
    print("Проверка смены пользователя")
    booking1.change_user(bob)
    print("booking1 пользователь:", booking1.user.name)  # Боб

    # Вернём обратно для чистоты теста
    booking1.change_user(alice)
    booking1.booking_status = True

    print()
    print("Проверка обмена бронями")

    print("До обмена:")
    print("booking1:", booking1.user.name, booking1.total_price(), booking1.user.bonus_points)
    print("booking2:", booking2.user.name, booking2.total_price(), booking2.user.bonus_points)

    result = exchange_bookings(booking1, booking2)

    print()
    print("Результат обмена:", result)

    print()
    print("После обмена:")
    print("booking1:", booking1.user.name, booking1.user.bonus_points)
    print("booking2:", booking2.user.name, booking2.user.bonus_points)

    print()
    print("Пример успешного обмена")
    alice = User(id=1, name="Алиса", rating=4.8, bonus_points=5000)
    bob = User(id=2, name="Боб", rating=4.2, bonus_points=200)
    desk = Address(city="Москва", street="Тверская, 15", floor=3)
    office = Address(city="Москва", street="Тверская, 15", floor=5)
    booking1 = Booking(
        address=desk,
        user=alice,
        workspace_type=WorkspaceType.DESK,
        price_per_hour=300,
        date=datetime.date(2026, 3, 25),
        hours=4
    )
    
    booking2 = Booking(
        address=office,
        user=bob,
        workspace_type=WorkspaceType.PRIVATE_OFFICE,
        price_per_hour=1200,
        date=datetime.date(2026, 3, 25),
        hours=3
    )


    print("До обмена:")
    print("booking1:", booking1.user.name, booking1.user.bonus_points)
    print("booking2:", booking2.user.name, booking2.user.bonus_points)

    result = exchange_bookings(booking1, booking2)

    print()
    print("Результат обмена:", result)

    print()
    print("После обмена:")
    print("booking1:", booking1.user.name, booking1.user.bonus_points)
    print("booking2:", booking2.user.name, booking2.user.bonus_points)


Проверка отмены брони
booking1 статус: False

Проверка смены пользователя
booking1 пользователь: Боб

Проверка обмена бронями
До обмена:
booking1: Алиса 1200 500
booking2: Боб 3600 200

Результат обмена: False

После обмена:
booking1: Алиса 500
booking2: Боб 200

Пример успешного обмена
До обмена:
booking1: Алиса 5000
booking2: Боб 200

Результат обмена: True

После обмена:
booking1: Боб 2600
booking2: Алиса 2600
